# Поиск общих ключевых точек на изображениях одного вида организмов

**Объект:** *Bosmina longirostris* (6 изображений, разные положение, масштаб, ориентация и качество съёмки).

**Задача:** найти на каждом изображении ключевые точки и дескрипторы, сопоставить их между изображениями, отфильтровать ненадёжные совпадения и выделить точки, которые прослеживаются на всех изображениях сразу. Одинаковая точка на разных снимках получает одинаковый номер.

**План ноутбука**

| № | Раздел | Что делаем |
|---|--------|-----------|
| 1 | Окружение | Установка `opencv-contrib`, импорты |
| 2 | Данные | Загрузка, метаданные, статистика яркости, контраста и резкости |
| 3 | Препроцессинг | Приведение к одному масштабу, три варианта контраста (raw / CLAHE mid / CLAHE hard) |
| 4 | SIFT | Параметры, детекция, влияние препроцессинга, особый случай изображения №6 |
| 5 | Сегментация | Маска организма, отсечение точек на фоне |
| 6 | Базовый матчинг | kNN + ratio test, без геометрической проверки |
| 7 | Сравнение детекторов | SIFT default / SIFT tuned / AKAZE на одной паре |
| 8 | Общие точки | Единый конвейер: ratio test → RANSAC/LMedS → маска → фильтр по расстоянию дескрипторов → треки |
| 9 | Итоговая статистика | Сколько точек найдено, сколько совпадений, сколько общих |
| 10 | Исключённые подходы | Что пробовалось и почему убрано |
| 11 | Выводы | |

> **Важно.** Часть методов из задания (SIFT, AKAZE) рассчитана на жёсткие сцены. Здесь объект живой, полупрозрачный, снят в разных ракурсах, поэтому точное сопоставление точек заранее невозможно. Цель работы — изучить параметры методов и честно оценить, что они дают.

## 1. Окружение

Используется `opencv-contrib-python==4.10.0.84`. Контрибный пакет нужен для модуля `xfeatures2d` и гарантирует, что `cv2.SIFT_create` и `cv2.AKAZE_create` доступны в одной сборке. Установка выполняется в Google Colab; в других средах ячейку с `pip` можно пропустить.

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip install opencv-contrib-python==4.10.0.84

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from tqdm import tqdm
from google.colab import drive

print("OpenCV:", cv2.__version__)

## 2. Данные

Папка с изображениями лежит на Google Drive (`MyDrive/PLANKTON`). Ищем файлы рекурсивно по основным расширениям, сортируем по имени, чтобы нумерация изображений №1–№6 была воспроизводимой.

In [ ]:
drive.mount('/content/drive')
IMAGE_DIR = "/content/drive/MyDrive/PLANKTON"
assert os.path.exists(IMAGE_DIR), f"Папка не найдена: {IMAGE_DIR}"

extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff")
image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, "**", ext.upper()), recursive=True))
image_paths = sorted(set(image_paths))

print(f"Найдено изображений: {len(image_paths)}")
for i, p in enumerate(image_paths, 1):
    print(f"  {i}. {os.path.relpath(p, IMAGE_DIR)}")

In [ ]:
records = []
for i, path in enumerate(image_paths, 1):
    img = cv2.imread(path)
    ok = img is not None
    records.append({
        "id": i,
        "filename": os.path.basename(path),
        "width":  img.shape[1] if ok else None,
        "height": img.shape[0] if ok else None,
        "channels": (img.shape[2] if img.ndim == 3 else 1) if ok else None,
        "filesize_kb": round(os.path.getsize(path) / 1024, 1),
    })

df = pd.DataFrame(records)
display(df)

### 2.1 Загрузка и предварительный просмотр

Для каждого изображения хранятся три версии: BGR (как читает OpenCV), RGB (для `matplotlib`) и градации серого (для детекторов; SIFT и AKAZE работают с одним каналом).

In [ ]:
images_bgr, images_gray, images_rgb = [], [], []
for path in tqdm(image_paths, desc="Загрузка"):
    bgr = cv2.imread(path)
    assert bgr is not None, f"Не удалось прочитать {path}"
    images_bgr.append(bgr)
    images_gray.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY))
    images_rgb.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

N_IMG = len(images_gray)
print("Загружено изображений:", N_IMG)

cols = 3
rows = (N_IMG + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = np.atleast_1d(axes).flatten()
for i, (rgb, path) in enumerate(zip(images_rgb, image_paths)):
    h, w = rgb.shape[:2]
    axes[i].imshow(rgb)
    axes[i].set_title(f"{i+1}. {os.path.basename(path)}\n{w}×{h}", fontsize=10)
for ax in axes:
    ax.axis("off")
plt.suptitle("Bosmina longirostris — исходные изображения", fontsize=14)
plt.tight_layout()
plt.show()

### 2.2 Качество изображений

Считаем для каждого кадра метрики, от которых зависит число и качество ключевых точек:

- **mean_brightness** — средняя яркость (0–255);
- **std_contrast** — стандартное отклонение яркости, грубая мера контраста;
- **pct_overexposed / pct_underexposed** — доля пикселей ярче 250 и темнее 5 (пересвет и недосвет);
- **sharpness_var_lap** — дисперсия лапласиана, мера резкости (чем больше, тем резче).

In [ ]:
stats = []
for i, gray in enumerate(images_gray, 1):
    h, w = gray.shape
    stats.append({
        "id": i,
        "size": f"{w}×{h}",
        "megapixels": round(w * h / 1e6, 2),
        "mean_brightness": round(float(gray.mean()), 1),
        "std_contrast": round(float(gray.std()), 1),
        "pct_overexposed": round(float((gray > 250).mean() * 100), 2),
        "pct_underexposed": round(float((gray < 5).mean() * 100), 2),
        "sharpness_var_lap": round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 1),
    })
stats_df = pd.DataFrame(stats)
display(stats_df)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (i, gray) in zip(axes.flatten(), enumerate(images_gray, 1)):
    ax.hist(gray.ravel(), bins=64, range=(0, 255), color="steelblue", alpha=0.8)
    ax.set_title(f"№{i}: mean={gray.mean():.0f}, std={gray.std():.0f}", fontsize=10)
    ax.set_xlabel("Яркость"); ax.set_ylabel("Пикселей"); ax.grid(alpha=0.3)
plt.suptitle("Гистограммы яркости (grayscale)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Препроцессинг

### 3.1 Параметры

**Масштаб.** Все изображения приводятся к длинной стороне `MAX_SIDE = 1280` px, чтобы детекторы работали в сопоставимом диапазоне масштабов:

- большие кадры уменьшаются через `cv2.INTER_AREA` (не создаёт муар при уменьшении);
- маленькие (изображение №6) увеличиваются через `cv2.INTER_CUBIC`.

**Контраст.** Три варианта входа:

| Вариант | Обработка | Параметры |
|---------|-----------|-----------|
| `raw` | только масштаб | — |
| `CLAHE mid` | локальная адаптивная эквализация | `clipLimit=2.0`, `tileGridSize=(8, 8)` |
| `CLAHE hard` | более сильная CLAHE + билатеральный фильтр | `clipLimit=4.0`, `tileGridSize=(16, 16)`; `bilateralFilter(d=5, sigmaColor=50, sigmaSpace=50)` |

`clipLimit` ограничивает усиление контраста (выше — сильнее, но растёт и шум); `tileGridSize` задаёт размер локальных блоков (больше плиток — более локальная эквализация). Билатеральный фильтр после сильной CLAHE подавляет усиленный шум, сохраняя границы.

In [ ]:
MAX_SIDE = 1280

def resize_to_max_side(gray, max_side=MAX_SIDE):
    # Приводит длинную сторону к max_side. Возвращает (изображение, коэффициент масштаба).
    h, w = gray.shape[:2]
    s = max_side / max(h, w)
    if s == 1.0:
        return gray, 1.0
    interp = cv2.INTER_AREA if s < 1.0 else cv2.INTER_CUBIC
    out = cv2.resize(gray, (int(round(w * s)), int(round(h * s))), interpolation=interp)
    return out, s

def apply_clahe(gray, clip, tile):
    return cv2.createCLAHE(clipLimit=clip, tileGridSize=tile).apply(gray)

gray_norm, scale_records = [], []
for i, g in enumerate(images_gray, 1):
    out, s = resize_to_max_side(g)
    gray_norm.append(out)
    scale_records.append({"id": i,
                          "orig": f"{g.shape[1]}×{g.shape[0]}",
                          "new":  f"{out.shape[1]}×{out.shape[0]}",
                          "scale": round(s, 3),
                          "direction": "upscale" if s > 1 else ("downscale" if s < 1 else "—")})
print(pd.DataFrame(scale_records).to_string(index=False))

gray_raw        = gray_norm
gray_clahe_mid  = [apply_clahe(g, 2.0, (8, 8)) for g in gray_norm]
gray_clahe_hard = [cv2.bilateralFilter(apply_clahe(g, 4.0, (16, 16)), d=5, sigmaColor=50, sigmaSpace=50)
                   for g in gray_norm]

fig, axes = plt.subplots(3, N_IMG, figsize=(3.6 * N_IMG, 10))
for k in range(N_IMG):
    for row, (imgs, title) in enumerate([(gray_raw, "raw"),
                                         (gray_clahe_mid, "CLAHE clip=2.0"),
                                         (gray_clahe_hard, "CLAHE clip=4.0 + bilateral")]):
        axes[row, k].imshow(imgs[k], cmap="gray", vmin=0, vmax=255)
        axes[row, k].set_title(f"{k+1}. {title}", fontsize=9)
        axes[row, k].axis("off")
plt.suptitle("Три версии препроцессинга", fontsize=13)
plt.tight_layout()
plt.show()

## 4. SIFT: параметры и детекция

### 4.1 Как устроен SIFT и что настраивается

SIFT строит пирамиду масштабов (октавы × слои), ищет экстремумы разности гауссианов (DoG), уточняет их положение, отбрасывает слабые и краевые отклики, назначает каждой точке ориентацию и считает дескриптор: вектор из 128 чисел (сетка 4×4 ячейки × 8 направлений градиента), сравнивается по норме **L2**.

| Параметр | Default | Что делает | Эффект при изменении |
|----------|---------|-----------|----------------------|
| `nfeatures` | 0 | Максимум точек (0 — без ограничения) | Оставляет лучшие по отклику |
| `nOctaveLayers` | 3 | Число слоёв в октаве | Больше слоёв — больше точек и больше устойчивость к масштабу, дольше счёт |
| `contrastThreshold` | 0.04 | Порог контраста DoG-отклика | **Меньше — больше слабых точек** (полезно для малоконтрастных объектов) |
| `edgeThreshold` | 10 | Порог отсечения краевых точек (отношение главных кривизн) | **Больше — больше точек на краях** (вытянутые контуры организма) |
| `sigma` | 1.6 | Сигма гауссова размытия на входе | Больше — сглаживает мелкие детали |

### 4.2 Используемые конфигурации

| Имя | `contrastThreshold` | `edgeThreshold` | `nOctaveLayers` | Где применяется |
|-----|--------------------|-----------------|-----------------|-----------------|
| `sift` (default) | 0.04 | 10 | 3 | изображения №1–5 в базовых экспериментах |
| `sift_soft` | 0.005 | 15 | 3 | изображение №6 (малоконтрастное; при default точек мало) |
| `sift_tuned` | 0.01 | 20 | 5 | основной конвейер общих точек (раздел 8) |

In [ ]:
sift      = cv2.SIFT_create()                                              # default
sift_soft = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=15)     # для №6
sift_tuned = cv2.SIFT_create(contrastThreshold=0.01, edgeThreshold=20, nOctaveLayers=5)

IDX_LOW_CONTRAST = 5   # изображение №6 (индекс с нуля)

def detect_sift(imgs, masks=None):
    # SIFT default для всех, кроме №6 (там sift_soft).
    kps, descs = [], []
    for i, im in enumerate(imgs):
        det = sift_soft if i == IDX_LOW_CONTRAST else sift
        kp, d = det.detectAndCompute(im, None if masks is None else masks[i])
        kps.append(kp); descs.append(d)
    return kps, descs

kps_raw,  descs_raw  = detect_sift(gray_raw)
kps_mid,  descs_mid  = detect_sift(gray_clahe_mid)
kps_hard, descs_hard = detect_sift(gray_clahe_hard)

print("Число ключевых точек по версиям препроцессинга:")
print(pd.DataFrame({"id": range(1, N_IMG + 1),
                    "raw": [len(k) for k in kps_raw],
                    "clahe_mid": [len(k) for k in kps_mid],
                    "clahe_hard": [len(k) for k in kps_hard]}).to_string(index=False))

### 4.3 Визуализация: как препроцессинг влияет на положение точек

Круг отображает масштаб точки (`DRAW_RICH_KEYPOINTS`), линия внутри — ориентацию.

In [ ]:
def draw_kp_grid(images_list, kps_list, row_labels, suptitle):
    n_rows, n_cols = len(images_list), len(images_list[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.2 * n_rows), squeeze=False)
    for r in range(n_rows):
        for c in range(n_cols):
            vis = cv2.drawKeypoints(images_list[r][c], kps_list[r][c], None,
                                    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, color=(0, 255, 0))
            h, w = images_list[r][c].shape
            axes[r, c].imshow(vis)
            axes[r, c].set_title(f"{c+1}. {row_labels[r]}\n{len(kps_list[r][c])} точек | {w}×{h}", fontsize=9)
            axes[r, c].axis("off")
    plt.suptitle(suptitle, fontsize=14)
    plt.tight_layout()
    plt.show()

draw_kp_grid([gray_raw, gray_clahe_mid, gray_clahe_hard],
             [kps_raw, kps_mid, kps_hard],
             ["raw", "CLAHE mid", "CLAHE hard"],
             "SIFT: raw vs CLAHE mid vs CLAHE hard (для №6 — sift_soft)")

In [ ]:
# Крупный план одного изображения
IDX = 1
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, (name, imgs, kps) in zip(axes, [("raw", gray_raw, kps_raw),
                                         ("CLAHE mid", gray_clahe_mid, kps_mid),
                                         ("CLAHE hard", gray_clahe_hard, kps_hard)]):
    vis = cv2.drawKeypoints(imgs[IDX], kps[IDX], None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, color=(0, 255, 0))
    ax.imshow(vis); ax.axis("off")
    ax.set_title(f"№{IDX+1} — {name} | {len(kps[IDX])} точек", fontsize=13)
plt.suptitle(f"Крупный план изображения №{IDX+1}", fontsize=14)
plt.tight_layout()
plt.show()

### 4.4 Размеры ключевых точек

`KeyPoint.size` — диаметр окрестности, по которой считается дескриптор (в пикселях). Медиана и перцентили показывают, на каком масштабе детали SIFT «видит» организм при каждом варианте препроцессинга.

In [ ]:
def kp_size_stats(kps, label):
    rows = []
    for i, kp_list in enumerate(kps, 1):
        sizes = np.array([k.size for k in kp_list]) if len(kp_list) else np.array([0.0])
        rows.append({"id": i, "label": label, "n": len(kp_list),
                     "median_size": round(float(np.median(sizes)), 1),
                     "p10": round(float(np.percentile(sizes, 10)), 1),
                     "p90": round(float(np.percentile(sizes, 90)), 1)})
    return pd.DataFrame(rows)

print("Размеры ключевых точек (px):")
print(pd.concat([kp_size_stats(kps_raw, "raw"),
                 kp_size_stats(kps_mid, "CLAHE mid"),
                 kp_size_stats(kps_hard, "CLAHE hard")], ignore_index=True).to_string(index=False))

### 4.5 Выбор препроцессинга для сопоставления

**Решение:** `CLAHE mid` для изображений №1–5, `raw` для №6.

- `CLAHE mid` повышает локальный контраст полупрозрачного тела, не раздувая шум так, как `CLAHE hard`.
- Изображение №6 — самое маленькое и малоконтрастное: локальная эквализация усиливает на нём шум, поэтому берётся `raw` (после увеличения до 1280 px) и мягкий `sift_soft`.

In [ ]:
imgs_final = [gray_raw[k] if k == IDX_LOW_CONTRAST else gray_clahe_mid[k] for k in range(N_IMG)]
kps_final, descs_final = detect_sift(imgs_final)

print("Финальное число ключевых точек:")
for k in range(N_IMG):
    src = "raw + sift_soft" if k == IDX_LOW_CONTRAST else "CLAHE mid + sift (default)"
    print(f"  №{k+1}: {len(kps_final[k]):>6} точек  ({src})")

fig, axes = plt.subplots(1, N_IMG, figsize=(3.6 * N_IMG, 4))
for k in range(N_IMG):
    vis = cv2.drawKeypoints(imgs_final[k], kps_final[k], None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, color=(0, 255, 0))
    axes[k].imshow(vis); axes[k].axis("off")
    axes[k].set_title(f"{k+1}. {len(kps_final[k])} точек", fontsize=10)
plt.suptitle("Финальный препроцессинг и SIFT-точки", fontsize=13)
plt.tight_layout()
plt.show()

### 4.6 Проверка выбора для изображения №6

Сравниваем на уже приведённом к 1280 px изображении №6 три варианта: SIFT default, `sift_soft` и `sift_soft` после билатеральной фильтрации (`d=9, sigmaColor=75, sigmaSpace=75`). Для оценки полезности точек считаем быстрый тест «№6 ↔ №1» и «№6 ↔ №3»: сколько совпадений проходит ratio test (0.75) и сколько из них остаётся после RANSAC-гомографии (порог 3 px, `maxIters=5000`, `confidence=0.999`).

Гомография здесь используется как грубый индикатор согласованности точек: чем больше inliers, тем лучше точки в этом варианте соответствуют друг другу геометрически.

In [ ]:
def knn_ratio_match(desc_a, desc_b, norm=cv2.NORM_L2, ratio=0.75):
    # kNN (k=2) + тест Лоу: оставляем совпадение, если оно заметно лучше второго кандидата.
    if desc_a is None or desc_b is None or len(desc_a) < 2 or len(desc_b) < 2:
        return []
    pairs = cv2.BFMatcher(norm, crossCheck=False).knnMatch(desc_a, desc_b, k=2)
    return [p[0] for p in pairs if len(p) == 2 and p[0].distance < ratio * p[1].distance]

def quick_inliers(desc_a, kp_a, desc_b, kp_b, ratio=0.75, thresh=3.0):
    # (число совпадений после ratio test, число inliers гомографии RANSAC)
    good = knn_ratio_match(desc_a, desc_b, cv2.NORM_L2, ratio)
    if len(good) < 8:
        return len(good), 0
    src = np.float32([kp_a[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp_b[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    _, mask = cv2.findHomography(src, dst, cv2.RANSAC, thresh, maxIters=5000, confidence=0.999)
    return len(good), int(mask.sum()) if mask is not None else 0

img6 = imgs_final[IDX_LOW_CONTRAST]
img6_dn = cv2.bilateralFilter(img6, d=9, sigmaColor=75, sigmaSpace=75)
variants = [("SIFT default",           sift,      img6),
            ("sift_soft",              sift_soft, img6),
            ("bilateral + sift_soft",  sift_soft, img6_dn)]

rows, kp_vis = [], []
for name, det, im in variants:
    kp, d = det.detectAndCompute(im, None)
    r1, i1 = quick_inliers(d, kp, descs_final[0], kps_final[0])
    r3, i3 = quick_inliers(d, kp, descs_final[2], kps_final[2])
    rows.append({"вариант №6": name, "точек": len(kp),
                 "vs №1 (ratio / inliers)": f"{r1} / {i1}",
                 "vs №3 (ratio / inliers)": f"{r3} / {i3}", "Σ inliers": i1 + i3})
    kp_vis.append((name, im, kp))
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, im, kp) in zip(axes, kp_vis):
    ax.imshow(cv2.drawKeypoints(im, kp, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, color=(0, 255, 0)))
    ax.set_title(f"{name} | {len(kp)} точек", fontsize=11); ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Сегментация организма

Фон в кадре почти однородный, и многие точки SIFT/AKAZE оказываются на шуме, пыли и границах кадра. Чтобы не сопоставлять фон, строим бинарную маску организма классическими средствами (без обучения).

### 5.1 Алгоритм `segment_organism`

1. Модуль отклонения яркости от медианы кадра `|I − median(I)|` (медиана ≈ яркость фона), нормировка в 0–255.
2. Порог по Отсу.
3. Морфология (эллиптические ядра): закрытие `21×21` (2 итерации) → открытие `11×11` (1 итерация).
4. Оставляем наибольшую связную компоненту (8-связность).
5. Заполнение дыр: закрытие `41×41` (3 итерации).
6. Опционально (`erode`) — эрозия ядром `15×15`, чтобы отодвинуть маску от границы тела: на самой границе много нестабильных точек.

### 5.2 Применение маски

Маска используется двумя способами:

- **на входе детектора** — `detectAndCompute(img, mask)` ищет точки только внутри организма;
- **на выходе** — отбрасываем совпадения, у которых хотя бы одна точка вне маски. Проверка `is_inside_mask`: среднее значение маски в окне `(2·margin+1)²` при `margin=5` больше 128 (точка не должна оказываться на границе).

In [ ]:
def segment_organism(gray, erode=0):
    # Бинарная маска организма (255 — организм, 0 — фон).
    med = float(np.median(gray))
    diff = np.abs(gray.astype(np.float32) - med)
    diff_n = np.clip(diff / (diff.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
    _, mask = cv2.threshold(diff_n, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open,  iterations=1)

    n_lab, labels, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n_lab > 1:
        largest = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
        mask = (labels == largest).astype(np.uint8) * 255

    k_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_fill, iterations=3)

    if erode > 0:
        mask = cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode, erode)), iterations=1)
    return mask

def is_inside_mask(kp, mask, margin=5):
    x, y = int(round(kp.pt[0])), int(round(kp.pt[1]))
    if not (0 <= y < mask.shape[0] and 0 <= x < mask.shape[1]):
        return False
    y0, y1 = max(0, y - margin), min(mask.shape[0], y + margin + 1)
    x0, x1 = max(0, x - margin), min(mask.shape[1], x + margin + 1)
    return mask[y0:y1, x0:x1].mean() > 128

masks_final = [segment_organism(im) for im in imgs_final]

fig, axes = plt.subplots(1, N_IMG, figsize=(3.6 * N_IMG, 4))
for k in range(N_IMG):
    vis = cv2.cvtColor(imgs_final[k], cv2.COLOR_GRAY2BGR)
    cnts, _ = cv2.findContours(masks_final[k], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cnts, -1, (0, 255, 0), 3)
    axes[k].imshow(vis[..., ::-1]); axes[k].axis("off")
    axes[k].set_title(f"{k+1}. покрытие {(masks_final[k] > 0).mean() * 100:.1f}%", fontsize=10)
plt.suptitle("Маска организма (зелёный контур)", fontsize=13)
plt.tight_layout()
plt.show()

### 5.3 Эффект маски на число точек SIFT

In [ ]:
kps_masked, descs_masked = detect_sift(imgs_final, masks=masks_final)

print(f"{'ID':<4}{'без маски':<12}{'с маской':<12}{'отсечено, %':<12}")
for k in range(N_IMG):
    b, a = len(kps_final[k]), len(kps_masked[k])
    print(f"{k+1:<4}{b:<12}{a:<12}{(1 - a / b) * 100 if b else 0:<12.1f}")

fig, axes = plt.subplots(1, N_IMG, figsize=(3.6 * N_IMG, 4))
for k in range(N_IMG):
    vis = cv2.drawKeypoints(imgs_final[k], kps_masked[k], None,
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, color=(0, 255, 0))
    axes[k].imshow(vis); axes[k].axis("off")
    axes[k].set_title(f"{k+1}. {len(kps_masked[k])} точек", fontsize=10)
plt.suptitle("SIFT только внутри маски организма", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Базовое сопоставление дескрипторов (без геометрии)

### 6.1 Параметры

- **Matcher:** `cv2.BFMatcher(NORM_L2, crossCheck=False)` — полный перебор; для SIFT (вещественные дескрипторы) используется L2. Для бинарных дескрипторов (AKAZE) далее берётся `NORM_HAMMING`.
- **kNN:** `k=2` — для каждого дескриптора эталона находятся два ближайших в целевом изображении.
- **Тест Лоу (ratio test):** совпадение принимается, если `d1 < ratio · d2`. Значение `ratio = 0.75` — типичный компромисс: меньше — строже (меньше ложных, но и верных), больше — мягче.

Эталон — изображение №1 (`REF_IDX = 0`). Все остальные сопоставляются с ним. Показаны 50 лучших (по расстоянию) совпадений. На этом этапе ложные соответствия ещё не отфильтрованы геометрией, поэтому визуально видны линии, идущие в разные стороны.

In [ ]:
REF_IDX = 0

def visualize_matches(img1, kp1, img2, kp2, matches, title, max_draw=50, figsize=(18, 8)):
    if not matches:
        print(f"Нет совпадений для визуализации: {title}")
        return
    best = sorted(matches, key=lambda m: m.distance)[:max_draw]
    vis = cv2.drawMatches(img1, kp1, img2, kp2, best, None,
                          matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=figsize)
    plt.imshow(vis); plt.title(title, fontsize=14); plt.axis("off")
    plt.tight_layout(); plt.show()

match_results = {}
print(f"Сопоставление эталона (№{REF_IDX+1}) с остальными, SIFT, ratio=0.75:")
for t in range(N_IMG):
    if t == REF_IDX:
        continue
    match_results[t] = knn_ratio_match(descs_final[REF_IDX], descs_final[t], cv2.NORM_L2, 0.75)
    print(f"  №{REF_IDX+1} vs №{t+1}: {len(match_results[t])} совпадений (до геометрической проверки)")

for t in (1, 3, 5):
    visualize_matches(imgs_final[REF_IDX], kps_final[REF_IDX], imgs_final[t], kps_final[t],
                      match_results[t],
                      f"№{REF_IDX+1} vs №{t+1}: {len(match_results[t])} совпадений (показаны 50 лучших)")

## 7. Сравнение детекторов на одной паре (№1 ↔ №3)

Три конфигурации сравниваются на одной и той же паре с одинаковым ratio test (0.75):

| Конфигурация | Параметры | Дескриптор / метрика |
|--------------|-----------|----------------------|
| SIFT default | `contrastThreshold=0.04`, `edgeThreshold=10`, `nOctaveLayers=3` | 128 float, L2 |
| SIFT tuned | `contrastThreshold=0.01`, `edgeThreshold=20`, `nOctaveLayers=5` | 128 float, L2 |
| AKAZE | `threshold=0.001`, `nOctaves=4`, `nOctaveLayers=4`; остальное по умолчанию | MLDB (бинарный), Hamming |

**AKAZE:** детектор по гессиану в нелинейном масштабном пространстве (диффузия сохраняет границы, в отличие от гауссова размытия SIFT). Ключевые параметры: `threshold` — порог отклика детектора (меньше — больше точек), `nOctaves` и `nOctaveLayers` — размер масштабной пирамиды, `diffusivity` — функция диффузии (по умолчанию `KAZE_DIFF_PM_G2`), `descriptor_type` — тип дескриптора (по умолчанию MLDB), `descriptor_size` (0 — полный, 486 бит) и `descriptor_channels` (по умолчанию 3).

In [ ]:
akaze_cmp = cv2.AKAZE_create(threshold=0.001, nOctaves=4, nOctaveLayers=4)

def extract_features(images, detector, name):
    kps, descs = [], []
    for im in images:
        kp, d = detector.detectAndCompute(im, None)
        kps.append(kp); descs.append(d)
    print(f"  [{name}] " + ", ".join(f"№{i+1}: {len(k)}" for i, k in enumerate(kps)))
    return kps, descs

PAIR = (0, 2)
configs = [("SIFT default", sift, cv2.NORM_L2),
           ("SIFT tuned (contrast=0.01, edge=20, layers=5)", sift_tuned, cv2.NORM_L2),
           ("AKAZE (threshold=0.001)", akaze_cmp, cv2.NORM_HAMMING)]

print(f"Сравнение на паре №{PAIR[0]+1} vs №{PAIR[1]+1}")
summary = []
for name, det, norm in configs:
    kp_c, d_c = extract_features(imgs_final, det, name)
    good = knn_ratio_match(d_c[PAIR[0]], d_c[PAIR[1]], norm, 0.75)
    summary.append({"метод": name, "точек на №1": len(kp_c[PAIR[0]]),
                    "точек на №3": len(kp_c[PAIR[1]]), "совпадений (ratio 0.75)": len(good)})
    visualize_matches(imgs_final[PAIR[0]], kp_c[PAIR[0]], imgs_final[PAIR[1]], kp_c[PAIR[1]],
                      good, f"{name} | после ratio test: {len(good)}", max_draw=40, figsize=(16, 7))
print(pd.DataFrame(summary).to_string(index=False))

## 8. Общие ключевые точки на всех изображениях

### 8.1 Схема конвейера

Для эталона (№1) и каждого из остальных изображений:

1. **Детекция + дескрипторы** (SIFT tuned или AKAZE).
2. **kNN + ratio test** (`k=2`, `ratio=0.8`).
3. **Геометрическая фильтрация.** Оценивается фундаментальная матрица `findFundamentalMat`; совпадения, не согласованные с моделью, отбрасываются (outliers). Используются два робастных метода:
   - `FM_RANSAC` — случайная выборка; порог `ransacReprojThreshold` задаёт максимальное расстояние до эпиполярной линии, при котором совпадение считается inlier;
   - `FM_LMEDS` — минимизация медианы ошибки; не требует порога, устойчив до 50% outliers.
4. **Фильтр по маске организма** — обе точки совпадения должны лежать внутри маски (`is_inside_mask`).
5. **Сборка треков.** Для каждой точки эталона собирается словарь `{номер изображения → индекс точки}`; если для одного изображения нашлось несколько кандидатов, остаётся с минимальным расстоянием дескрипторов.
6. **Фильтр по среднему расстоянию дескрипторов** трека (`max_dist`).
7. **Ранжирование:** сначала по числу изображений, на которых найдена точка (убывание), затем по среднему расстоянию дескрипторов (возрастание). Берутся лучшие 30. Точка № *k* на эталоне и точка № *k* на остальных изображениях — одна и та же.

### 8.2 Параметры двух вариантов

| Параметр | **A: SIFT** | **B: AKAZE** |
|----------|-------------|--------------|
| Детектор | `SIFT_create(contrastThreshold=0.01, edgeThreshold=20, nOctaveLayers=5)`; для №6 — `sift_soft` | `AKAZE_create(threshold=1e-5, descriptor_size=0, descriptor_channels=3, nOctaves=4, nOctaveLayers=4, diffusivity=KAZE_DIFF_PM_G2)` |
| Метрика | L2 | Hamming |
| Маска при детекции | нет | да (маска с эрозией `15×15`) |
| `ratio` | 0.8 | 0.8 |
| Геометрия | `FM_LMEDS`, порог 3.0, `confidence=0.99`, `maxIters=5000` | `FM_RANSAC`, порог 6.0, `confidence=0.99`, `maxIters=10000` |
| Фильтр по маске после матчинга | да (маска без эрозии) | да (маска с эрозией, та же, что на входе) |
| `max_dist` (среднее расстояние дескрипторов) | 150 | 80 |
| Размер топа | 30 | 30 |

> Порог `threshold=1e-5` у AKAZE намного ниже значения по умолчанию (`1e-3`): у малоконтрастного объекта при стандартном пороге точек слишком мало, поэтому детектор намеренно делается «жадным», а надёжность обеспечивают последующие фильтры.

In [ ]:
def detect_features(imgs, detectors, masks=None):
    # detectors: один детектор или список по числу изображений
    kps, descs = [], []
    for i, im in enumerate(imgs):
        det = detectors[i] if isinstance(detectors, (list, tuple)) else detectors
        kp, d = det.detectAndCompute(im, None if masks is None else masks[i])
        kps.append(kp); descs.append(d)
    return kps, descs

def geometric_filter(kp_a, kp_b, matches, method, thresh, conf, iters):
    # Фильтрует совпадения по фундаментальной матрице (RANSAC или LMedS).
    if len(matches) < 8:
        return []
    src = np.float32([kp_a[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp_b[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    _, mask = cv2.findFundamentalMat(src, dst, method, thresh, conf, iters)
    if mask is None:
        return []
    mask = mask.ravel().astype(bool)
    return [m for m, ok in zip(matches, mask) if ok]

def build_tracks(kps, descs, norm, ratio, geom, mask_filter_masks, max_dist, ref_idx=REF_IDX):
    # Возвращает (tracks, dists, pair_stats).
    # tracks[ref_kp_idx] = {target_img: target_kp_idx}; dists[ref_kp_idx] = {target_img: distance}
    tracks, dists, pair_stats = defaultdict(dict), defaultdict(dict), []
    for t in range(len(kps)):
        if t == ref_idx:
            continue
        good = knn_ratio_match(descs[ref_idx], descs[t], norm, ratio)
        inl = geometric_filter(kps[ref_idx], kps[t], good, **geom)
        if mask_filter_masks is not None:
            inl_m = [m for m in inl
                     if is_inside_mask(kps[ref_idx][m.queryIdx], mask_filter_masks[ref_idx])
                     and is_inside_mask(kps[t][m.trainIdx], mask_filter_masks[t])]
        else:
            inl_m = inl
        pair_stats.append({"пара": f"№{ref_idx+1}–№{t+1}", "ratio test": len(good),
                           "после геометрии": len(inl), "после маски": len(inl_m)})
        for m in inl_m:
            r = m.queryIdx
            if t not in dists[r] or m.distance < dists[r][t]:
                tracks[r][t] = m.trainIdx
                dists[r][t] = m.distance
    valid = {r: tr for r, tr in tracks.items() if np.mean(list(dists[r].values())) < max_dist}
    ranked = sorted(valid.items(), key=lambda it: (-len(it[1]), np.mean(list(dists[it[0]].values()))))
    return tracks, dists, valid, ranked, pair_stats

def draw_tracks(imgs, kps, top_tracks, ref_idx, suptitle, cols=3):
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7.3 * cols, 7 * rows))
    axes = np.atleast_1d(axes).flatten()
    for i, im in enumerate(imgs):
        vis = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
        for track_id, (ref_kp, tr) in enumerate(top_tracks, 1):
            if i == ref_idx:
                kp_idx = ref_kp
            elif i in tr:
                kp_idx = tr[i]
            else:
                continue
            x, y = map(lambda v: int(round(v)), kps[i][kp_idx].pt)
            cv2.circle(vis, (x, y), 14, (0, 0, 255), 2)
            cv2.putText(vis, str(track_id), (x - 10, y + 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        axes[i].imshow(vis[..., ::-1]); axes[i].axis("off")
        axes[i].set_title(f"№{i+1}" + (" (ЭТАЛОН)" if i == ref_idx else ""),
                          fontsize=14, fontweight="bold" if i == ref_idx else "normal")
    for ax in axes[len(imgs):]:
        ax.axis("off")
    plt.suptitle(suptitle, fontsize=17)
    plt.tight_layout()
    plt.show()

def run_pipeline(name, kps, descs, norm, ratio, geom, mask_filter_masks, max_dist, imgs, top_n=30):
    tracks, dists, valid, ranked, pair_stats = build_tracks(kps, descs, norm, ratio, geom, mask_filter_masks, max_dist)
    top = ranked[:top_n]
    n_all = sum(1 for _, tr in valid.items() if len(tr) == len(imgs) - 1)

    print(f"=== {name} ===")
    print("Точек на изображениях:", ", ".join(f"№{i+1}: {len(k)}" for i, k in enumerate(kps)))
    print(pd.DataFrame(pair_stats).to_string(index=False))
    print(f"Треков-кандидатов (≥2 изображений): {len(tracks)}")
    print(f"После фильтра по расстоянию (<{max_dist}): {len(valid)}")
    print(f"Общих точек на ВСЕХ {len(imgs)} изображениях: {n_all}")
    print("Топ-30 по числу изображений:",
          dict(sorted(Counter(len(tr) + 1 for _, tr in top).items(), reverse=True)))
    draw_tracks(imgs, kps, top, REF_IDX, f"{name}: топ-{len(top)} общих точек (одинаковый номер = одна точка)")
    return {"name": name, "kps": kps, "pair_stats": pair_stats, "n_candidates": len(tracks),
            "n_valid": len(valid), "n_all": n_all, "top": top, "valid": valid}

### 8.3 Вариант A: SIFT + LMedS + маска + фильтр по расстоянию

In [ ]:
# Для №6 берём sift_soft; изображение уже приведено к 1280 px в разделе 3
detectors_A = [sift_soft if i == IDX_LOW_CONTRAST else sift_tuned for i in range(N_IMG)]
kps_A, descs_A = detect_features(imgs_final, detectors_A)

res_A = run_pipeline(
    "A: SIFT tuned + LMedS + маска", kps_A, descs_A, cv2.NORM_L2, ratio=0.8,
    geom=dict(method=cv2.FM_LMEDS, thresh=3.0, conf=0.99, iters=5000),
    mask_filter_masks=masks_final, max_dist=150, imgs=imgs_final)

### 8.4 Вариант B: AKAZE с маской на входе + RANSAC + фильтр по расстоянию

Маска строится с эрозией `15×15` и подаётся прямо в детектор: точки на границе тела и фоне не создаются вообще.

In [ ]:
akaze = cv2.AKAZE_create(threshold=0.00001, descriptor_size=0, descriptor_channels=3,
                         nOctaves=4, nOctaveLayers=4, diffusivity=cv2.KAZE_DIFF_PM_G2)

masks_eroded = [segment_organism(im, erode=15) for im in imgs_final]
print("Покрытие масок с эрозией:", ", ".join(f"№{i+1}: {(m > 0).mean() * 100:.1f}%" for i, m in enumerate(masks_eroded)))

kps_B, descs_B = detect_features(imgs_final, akaze, masks=masks_eroded)

res_B = run_pipeline(
    "B: AKAZE (маска на входе) + RANSAC", kps_B, descs_B, cv2.NORM_HAMMING, ratio=0.8,
    geom=dict(method=cv2.FM_RANSAC, thresh=6.0, conf=0.99, iters=10000),
    mask_filter_masks=masks_eroded, max_dist=80, imgs=imgs_final)

## 9. Итоговая статистика

Сводная таблица для отчёта: число найденных ключевых точек, число совпадений на каждом этапе фильтрации, число общих точек (на всех 6 изображениях) и число «частично общих» (на ≥2 изображениях).

In [ ]:
rows = []
for res in (res_A, res_B):
    ps = pd.DataFrame(res["pair_stats"])
    rows.append({
        "метод": res["name"],
        "точек на изображениях": " / ".join(str(len(k)) for k in res["kps"]),
        "совпадений ratio (Σ по парам)": int(ps["ratio test"].sum()),
        "после геометрии (Σ)": int(ps["после геометрии"].sum()),
        "после маски (Σ)": int(ps["после маски"].sum()),
        "треков (≥2 изобр.)": res["n_candidates"],
        "после фильтра расстояния": res["n_valid"],
        f"общих на всех {N_IMG}": res["n_all"],
    })
summary_df = pd.DataFrame(rows)
display(summary_df.T)

## 10. Исключённые подходы

В ходе работы пробовались и убраны из итогового ноутбука следующие методы. Причина — не «не повезло с параметрами», а несоответствие метода данным.

| Подход | Почему не подходит |
|--------|--------------------|
| **Оптический поток Lucas–Kanade** (`calcOpticalFlowPyrLK`, окно 21×21, 3 уровня пирамиды) | Метод предполагает постоянство яркости и **малое смещение** между кадрами (видеопоследовательность). Здесь это разные фотографии с разным масштабом, поворотом и позой организма: смещения в десятки–сотни пикселей, ориентация различается. Поток «сносит» точки на ближайшие структуры, а не на соответствующие. Цепочка треков №1→№2→…→№6 дополнительно накапливает ошибку. |
| **LOGOS** (`xfeatures2d.matchLOGOS`) и **GMS** (`matchGMS`) | Фильтры используют идею пространственной согласованности соседних совпадений при близкой геометрии кадров (жёсткая сцена, небольшой параллакс). У полупрозрачного живого организма, снятого в разных ракурсах, соседние точки смещаются несогласованно, поэтому фильтры отбрасывают верные и пропускают случайные совпадения. Кроме того, в коде использовались «тихие» откаты на RANSAC при ошибке, что смешивало результаты разных методов. |
| **AKAZE + RANSAC без маски**, отдельный фильтр расстояния (`<120`) | Это промежуточные шаги варианта B: учтены в нём (маска на входе + фильтр `<80`), поэтому отдельно не приводятся. |
| **Повторное увеличение №6 до 1280 px** | Изображение уже приводилось к 1280 px в разделе 3, дополнительный `resize` не имел эффекта. |

> Фундаментальная матрица (используемая в разделе 8) предполагает жёсткую сцену, что для живого организма выполняется лишь приближённо. Поэтому такие фильтры стоит рассматривать как грубую отбраковку ложных совпадений, а не как строгое геометрическое доказательство соответствия.

## 11. Выводы



**1. Количество точек.** Число найденных ключевых точек сильно зависит от изображения. Самые «богатые» кадры — крупные, резкие и с хорошим контрастом; самое маленькое и малоконтрастное изображение №6 при стандартных параметрах SIFT даёт слишком мало точек, поэтому для него снижен `contrastThreshold` (0.005) и увеличен `edgeThreshold` (15). Умеренная CLAHE (`clip=2.0`) помогла выявить детали полупрозрачного тела, а сильная CLAHE с билатеральным фильтром усиливала и шум.

**2. Сопоставление.** Ratio test оставляет заметную долю совпадений, но без геометрической проверки среди них много случайных: линии на визуализации пересекаются и ведут в разные части тела. После `FM_LMEDS` / `FM_RANSAC` и фильтра по маске их количество сокращается — часть исходных совпадений оказывается ложной.

**3. Общие точки.** на деле общих точек на всех 6 снимках ноль у обоих методов. Максимум, что нашлось, это 9 точек на трёх снимках из шести (7 у SIFT и 2 у AKAZE). Причина — суммарный эффект ограничений: на каждом шаге теряется часть совпадений, а для трека на всех кадрах необходимо, чтобы точка выжила на каждой из пяти пар с эталоном.

**4. Где лежат устойчивые точки.** у SIFT точки на краю панциря, у AKAZE внутри тела.

**5. Устойчивость к изменениям.**
- *Масштаб и поворот в плоскости:* SIFT и AKAZE устойчивы к ним по построению (масштабное пространство и назначение ориентации); приведение к единому размеру дополнительно снижает разброс.
- *Положение и ракурс организма:* повороты вне плоскости и разная поза деформируют вид одной и той же детали, дескриптор меняется, и соответствие теряется. Это главное ограничение метода на данных такого типа.
- *Качество изображения:* низкое разрешение, малый контраст и шум снижают число точек и ухудшают дескрипторы (особенно №6).

**6. Сравнение методов.** SIFT (вещественный 128-мерный дескриптор, L2) обычно даёт более информативные дескрипторы, AKAZE (бинарный MLDB, Hamming) быстрее и лучше сохраняет границы благодаря нелинейной диффузии. Для малоконтрастного прозрачного объекта критичны пороги детекции: без их снижения точек не хватает, а с их снижением растёт доля ложных, которую приходится убирать последующими фильтрами. Оптический поток, LOGOS и GMS для данной задачи не подходят (раздел 10).

**7. Итог.** Классические дескрипторы не позволяют надёжно находить одни и те же анатомические точки у живых организмов, снятых в разных позах и условиях. Для строгого решения нужны методы, учитывающие нежёсткую деформацию или обучаемые признаки (например, обучаемое сопоставление, ключевые точки на основе нейросетей или ручная разметка ориентиров), либо съёмка в стандартизированном положении.